# Image Generation Model Evaluation

This notebook evaluates different image generation models for gift card illustrations.

## Metrics Tracked
- **latency_ms**: Time per image generation
- **output_size_bytes**: Size of generated image
- **quality_score**: LLM-as-a-judge + aesthetic scoring
- **cost_per_image**: Cost per image generation
- **monthly_estimate**: Cost for 2B images

## Models Evaluated
- Recraft API
- TokenFactory / Flux API
- SDXL Lightning (self-hosted)
- Flux (self-hosted)

In [1]:
import os
import re
import sys
from pathlib import Path

import mlflow
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "prototype" / "src"))

from src.evaluation import (
    evaluate_quality_with_judge,
    setup_mlflow,
)
from src.prompts import (
    generate_image_quality_judge_prompt,
)
from src.test_samples import get_test_profiles, get_test_recommendations_for_image
from src.utils import download_image_for_mlflow, load_env_from_repo_root

# Load .env file from repository root
load_env_from_repo_root()

In [2]:
setup_mlflow("image_generation_eval")

MLflow tracking URI: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud
MLflow experiment: <Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1765816864158, experiment_id='2', last_update_time=1765816864158, lifecycle_stage='active', name='image_generation_eval', tags={}>


In [5]:
from src.image_model_config import get_available_image_models

models_to_evaluate = get_available_image_models(
    include_recraft=True,
    include_tokenfactory=True,
    include_self_hosted=False,
)

models_to_evaluate

## Test All Models

Generate a sample image for each available model to verify connectivity and functionality.


In [6]:
# Test prompt for all models
test_prompt = """A magical, festive holiday card illustration featuring:
- A cheerful, age-appropriate scene for a 10-year-old child
- A toy plane in the center of the image (the wished gift)
- Holiday decorations: Christmas tree, snow, presents, stars
- Warm, colorful, and joyful atmosphere
- Suitable for a holiday gift card
- Style: whimsical, child-friendly, magical
- No text in the image"""

print(f"Testing {len(models_to_evaluate)} image generation models...\n")
print("All models use unified API: client.generate(prompt)")
print("Defaults: 1024x1024, PNG format, service-appropriate settings\n")

results = []

for model_config in models_to_evaluate:
    model_name = model_config.name
    client = model_config.client
    # Get service name from the model config
    model_dict = model_config.to_dict()
    service = model_dict["service"]

    print(f"🖼️  Testing {model_name} ({service})...")

    try:
        # Unified API: all services use the same simple call
        image_data, metrics = client.generate(prompt=test_prompt)

        # Determine if image_data is URL or base64
        is_url = isinstance(image_data, str) and image_data.startswith("http")
        is_base64 = isinstance(image_data, str) and len(image_data) > 100 and not is_url

        result = {
            "model": model_name,
            "service": service,
            "status": "✅ Success",
            "latency_ms": round(metrics.get("latency_ms", 0), 2),
            "image_type": "URL" if is_url else ("base64" if is_base64 else "unknown"),
            "image_preview": image_data[:50] + "..." if is_url or is_base64 else str(image_data)[:50],
        }

        if is_url:
            result["image_url"] = image_data
        elif is_base64:
            result["image_data_length"] = len(image_data)

        results.append(result)
        print(f"   ✅ Generated successfully (latency: {result['latency_ms']}ms)")
        if is_url:
            print(f"   📷 Image URL: {image_data}")
        elif is_base64:
            print(f"   📷 Image: base64 encoded ({len(image_data)} chars)")

    except Exception as e:
        error_msg = str(e)
        result = {
            "model": model_name,
            "service": service,
            "status": "❌ Failed",
            "error": error_msg[:100] + "..." if len(error_msg) > 100 else error_msg,
        }
        results.append(result)
        print(f"   ❌ Error: {error_msg[:80]}...")

    print()

# Display summary
print("=" * 60)
print("Test Summary:")
print("=" * 60)
df_test = pd.DataFrame(results)
print(df_test[["model", "service", "status", "latency_ms"]].to_string(index=False))
print()

# Show successful models
successful = [r for r in results if r["status"] == "✅ Success"]
print(f"✅ {len(successful)}/{len(results)} models working successfully")


Testing 3 image generation models...

All models use unified API: client.generate(prompt)
Defaults: 1024x1024, PNG format, service-appropriate settings

🖼️  Testing recraftv3 (recraft)...
   ✅ Generated successfully (latency: 4929.9ms)
   📷 Image URL: https://img.recraft.ai/VNH-QSvBD_Avy7JGG-igDCH1sQ1qjerg5xaGElDHmU0/rs:fit:1024:1024:0/raw:1/plain/abs://external/images/dbad2f67-1352-4b27-9e47-768957066601

🖼️  Testing tokenfactory-flux-schnell (tokenfactory)...
   ✅ Generated successfully (latency: 2327.46ms)
   📷 Image URL: https://pictures-storage.storage.eu-north1.nebius.cloud/text2img-3e712808-9721-4703-b966-db7802a9a40d_00001_.png

🖼️  Testing tokenfactory-flux-dev (tokenfactory)...
   ✅ Generated successfully (latency: 10764.23ms)
   📷 Image URL: https://pictures-storage.storage.eu-north1.nebius.cloud/text2img-6a8dfdee-6c34-4612-8c7d-c430411c34dc_00001_.png

Test Summary:
                    model      service    status  latency_ms
                recraftv3      recraft ✅ Success

In [ ]:
# import os
# from openai import OpenAI

# client = OpenAI(
#     base_url="https://api.tokenfactory.nebius.com/v1/",
#     api_key=os.environ.get("NEBIUS_API_KEY")
# )

# response = client.images.generate(
#     model="black-forest-labs/flux-schnell",
#     extra_body={
#         "response_extension": "png",
#         "width": 1024,
#         "height": 1024,
#         # "num_inference_steps": 30,
#         "seed": -1,
#         # "negative_prompt": "Giraffes, night sky"
#     },
#     # prompt="Create an image of an astronaut exploring Mars."
#     prompt = """A magical, festive holiday card illustration featuring:
# - A cheerful, age-appropriate scene for a 10-year-old child
# - An plane in the center of the image (the wished gift)
# - Holiday decorations: Christmas tree, snow, presents, stars
# - Warm, colorful, and joyful atmosphere
# - Suitable for a holiday gift card
# - Style: whimsical, child-friendly, magical
# - No text in the image"""
# )

# print(response.to_json())

{
  "data": [
    {
      "b64_json": null,
      "url": "https://pictures-storage.storage.eu-north1.nebius.cloud/text2img-c11650ba-59ed-4238-ba98-9748d0d6bfb7_00001_.png"
    }
  ],
  "id": "text2img-c11650ba-59ed-4238-ba98-9748d0d6bfb7"
}


In [7]:


# # client = ImageClient(
# #     base_url="https://api.tokenfactory.nebius.com/v1",
# #     api_key=os.environ.get("NEBIUS_API_KEY"),
# #     model="black-forest-labs/flux-schnell"
# # )

# for client in models_to_evaluate:
#     image_data, metrics = client.generate(
#     prompt = """A magical, festive holiday card illustration featuring:
#     - A cheerful, age-appropriate scene for a 10-year-old child
#     - An plane in the center of the image (the wished gift)
#     - Holiday decorations: Christmas tree, snow, presents, stars
#     - Warm, colorful, and joyful atmosphere
#     - Suitable for a holiday gift card
#     - Style: whimsical, child-friendly, magical
#     - No text in the image""",
#         size="1024x1024",
#         extra_body={
#             "response_extension": "png",
#             "width": 1024,
#             "height": 1024,
#             "seed": -1,
#         }
#     )
#     print(image_data)

In [8]:
from src.model_config import get_judge_client

judge_client = get_judge_client()

In [ ]:
from src.generators import generate_card_image

test_profiles = get_test_profiles()
test_recommendations = get_test_recommendations_for_image()

results = []
agg_results = []

MLFLOW_AVAILABLE = bool(mlflow.get_tracking_uri())
mlflow.autolog()


# Evaluate all configured models
for model_config in models_to_evaluate:
    # Handle both ImageModelConfig objects and dict format (backward compatibility)
    if hasattr(model_config, "name"):
        # ImageModelConfig object
        model_name = model_config.name
        client = model_config.client
        cost_per_image = model_config.cost_per_image
        model_dict = model_config.to_dict()
        service = model_dict["service"]
    else:
        # Dict format (backward compatibility)
        model_name = model_config["name"]
        service = model_config["service"]
        client = model_config["client"]
        cost_per_image = model_config["cost_per_image"]
    print(f"Evaluating model: {model_name}")

    with mlflow.start_run(run_name=f"{model_name}"):
        parent_run_id = mlflow.active_run().info.run_id if MLFLOW_AVAILABLE else None
        per_calls = []

        for kid_profile, gift_recommendation in zip(test_profiles, test_recommendations):
            try:

                # prompt = generate_image_prompt(kid_profile, gift_recommendation)
                image, metrics = generate_card_image(
                    image_client=client,
                    kid_profile=kid_profile,
                    gift_recommendation=gift_recommendation
                )

                quality_score = None
                quality_rationale = None
                if judge_client:
                    judge_prompt = generate_image_quality_judge_prompt(image.prompt, image.image_url, kid_profile)
                    quality_score, quality_rationale = evaluate_quality_with_judge(judge_client, judge_prompt)

                record = {
                    "model": image.model_version,
                    "service": service,
                    "kid_id": kid_profile.id,
                    "latency_ms": round(metrics.get("latency_ms", 0), 0),
                    "output_size_bytes": metrics.get("output_size_bytes"),
                    "quality_score": quality_score,
                    "cost_per_image": cost_per_image,
                    "cost_per_1m_images": cost_per_image * 1000000,
                }
                per_calls.append(record)
                results.append(record)

                if MLFLOW_AVAILABLE:
                    with mlflow.start_run(run_name=f"{kid_profile.id}", nested=True):
                        mlflow.log_metric("latency_ms", record["latency_ms"])
                        # mlflow.log_metric("output_size_bytes", metrics.get("output_size_bytes", 0) or 0)
                        mlflow.log_metric("quality_score", quality_score)
                        mlflow.log_metric("cost_per_image", cost_per_image)
                        mlflow.log_metric("cost_per_1m_images", cost_per_image * 1000000)

                        mlflow.set_tags({
                            "task": "image_generation",
                            "model": image.model_version,
                        })

                        # Download and log image as artifact
                        mlflow.log_param("image_url", image.image_url)
                        try:
                            image_path = download_image_for_mlflow(image.image_url)
                            mlflow.log_artifact(image_path, "generated_image")
                            # Clean up temp file if it was created
                            if image_path.startswith("/tmp") or image_path.startswith("/var"):
                                try:
                                    os.unlink(image_path)
                                except Exception:
                                    pass
                        except Exception as e:
                            print(f"Warning: Could not log image artifact: {e}")

                        # Log metadata as dict
                        mlflow.log_dict(
                            {
                                "prompt": image.prompt,
                                "quality_score": quality_score,
                                "quality_rationale": quality_rationale,
                            },
                            "image_evaluation.json"
                        )

            except Exception as e:
                err_rec = {
                    "model": model_name,
                    "service": service,
                    "kid_id": kid_profile.id,
                    "error": str(e),
                }
                per_calls.append(err_rec)
                results.append(err_rec)

        df_model = pd.DataFrame([r for r in per_calls if "error" not in r])
        if not df_model.empty:
            agg = {
                "latency_ms": round(df_model["latency_ms"].mean(), 0),
                "quality_score": df_model["quality_score"].mean(),
                "cost_per_image": df_model["cost_per_image"].mean(),
                "cost_per_1m_images": df_model["cost_per_1m_images"].mean(),
                "calls": len(df_model),
            }
            agg_results.append(agg)

            out_dir = Path("data/evaluation")
            out_dir.mkdir(parents=True, exist_ok=True)
            model_name_path = re.sub(r'[^\w\-]', '-', model_name)
            out_path = out_dir / f"04_image_eval-{model_name_path}.csv"
            df_model.to_csv(out_path, index=False)
            print(f"Saved per-call results for {model_name} -> {out_path}")

            if MLFLOW_AVAILABLE:
                mlflow.log_metric("latency_ms", agg["latency_ms"])
                mlflow.log_metric("quality_score", agg["quality_score"])
                mlflow.log_metric("cost_per_image", agg["cost_per_image"])
                mlflow.log_metric("cost_per_1m_images", agg["cost_per_1m_images"])
                mlflow.log_metric("calls", agg["calls"])
                mlflow.set_tags({"task": "image_generation"})
        else:
            print(f"No successful calls for model {model_name}")

2025/12/15 22:26:18 INFO mlflow.tracking.fluent: Autologging successfully enabled for openai.


Evaluating model: recraftv3
🏃 View run d4f14763-ef9e-4ac9-8aee-9772701233e0 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/2/runs/1cedc81c6a574903b4d8642afed30fc6
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/2
🏃 View run b7d241b6-118b-480a-9cce-8d2d62e96542 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/2/runs/5037378bdcf0451eb51039d4a3bcfd26
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/2
🏃 View run 78452c46-90e9-4b68-826b-97942d33e0b7 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/2/runs/4c85d57e037e4c2cb48f53dc4af3f266
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.m